# external entities



CoHDL can interact with modules written in other hardware description languages. The cohdl_yosys library also supports this and allows us to formally verify VHDL code.

In [1]:
# basic setup of jupyter notebook

from __future__ import annotations

import cohdl

# When an exception occurs during compilation,
# cohdl inserts fake stack frames into the exception traceback.
# This does not work properly inside jupyter notebooks.

cohdl.use_pretty_traceback(False)

The `Adder` entity is defined in the file `external/Adder.vhd`. To access it we need to describe its interface as an external CoHDL entity.

In [2]:
from cohdl import Port, Bit, Unsigned, std

# External cohdl entities define the interface of an
# entity implemented in a different HDL.
# They have no architecture method.
class Adder(cohdl.Entity, extern=True):
    clk = Port.input(Bit)

    value_a = Port.input(Unsigned[8])
    value_b = Port.input(Unsigned[8])

    result = Port.output(Unsigned[9])

We have to point Yosys to the external file path and can then verify it like any other CoHDL entity.

In [3]:
from cohdl_yosys import YosysTestCase, YosysParams

from cohdl_yosys.formal import When, always, prev, past_valid, set_default_ctx

class Adder_Formal(YosysTestCase, entity=Adder):
    _yosys_params_ = YosysParams(
        clean_build_dir=True,
        prove=True,
        quiet=True,
        # Pass VHDL file to yosys.
        # If the implementation has more dependencies they
        # must also be specified here.
        files=["external/Adder.vhd"],
    )

    def architecture(self, dut: Adder):

        set_default_ctx(clk=std.Clock(dut.clk))

        @std.concurrent
        def formal_properties():
            prev_a = prev(dut.value_a)
            prev_b = prev(dut.value_b)

            always["result_is_sum_of_inputs"](
                When(past_valid()).then_imm(
                    dut.result == prev_a.resize(9) + prev_b
                )
            )


if Adder_Formal().test_formal_properties(return_on_error=True):
    print("formal check has passed")
else:
    print("formal check has failed")

formal check has passed
